# 📈 04 — Revenue Forecasting
**E-Commerce Customer & Revenue Analytics Platform**

This notebook implements monthly revenue forecasting using machine learning:
- Aggregate monthly revenue time-series
- Engineer time and seasonal features
- Train Linear Regression (80/20 chronological split)
- Evaluate: MAE, MSE, RMSE, R²
- Visualise Actual vs Predicted
- Generate 6-month future forecast

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Step 1: Build Monthly Revenue Time-Series

In [ ]:
CLEAN_PATH = os.path.join('..', 'data', 'processed', 'cleaned_data.csv')
df = pd.read_csv(CLEAN_PATH, parse_dates=['InvoiceDate'])

monthly = (df.groupby('YearMonth')['Revenue']
             .sum().reset_index()
             .rename(columns={'Revenue': 'MonthlyRevenue'})
             .sort_values('YearMonth'))

monthly['TimeIndex'] = np.arange(len(monthly))
monthly['Month']     = pd.to_datetime(monthly['YearMonth']).dt.month
monthly['Quarter']   = pd.to_datetime(monthly['YearMonth']).dt.quarter
monthly['MonthSin']  = np.sin(2 * np.pi * monthly['Month'] / 12)
monthly['MonthCos']  = np.cos(2 * np.pi * monthly['Month'] / 12)

print(f'Time-series: {len(monthly)} monthly periods')
monthly.head()

In [ ]:
# Plot raw monthly revenue
fig, ax = plt.subplots(figsize=(13, 4))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')
ax.plot(monthly['YearMonth'], monthly['MonthlyRevenue'],
        color='#6C63FF', linewidth=2.2, marker='o', markersize=4)
ax.fill_between(monthly['YearMonth'], monthly['MonthlyRevenue'], alpha=0.12, color='#6C63FF')
ax.set_title('Monthly Revenue Time-Series', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Step 2: Feature Engineering & Train/Test Split

In [ ]:
feature_cols = ['TimeIndex', 'MonthSin', 'MonthCos', 'Quarter']
X = monthly[feature_cols].values
y = monthly['MonthlyRevenue'].values

split_idx = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train samples : {len(X_train)} months')
print(f'Test samples  : {len(X_test)} months')
print(f'Features used : {feature_cols}')

## Step 3: Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print('Model trained.')
print(f'Coefficients: {dict(zip(feature_cols, model.coef_.round(2)))}')
print(f'Intercept   : {model.intercept_:.2f}')

## Step 4: Evaluation Metrics

In [ ]:
y_pred_test = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred_test)
mse  = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred_test)

metrics = pd.DataFrame({
    'Metric': ['MAE', 'MSE', 'RMSE', 'R²'],
    'Value': [f'£{mae:,.2f}', f'{mse:,.2f}', f'£{rmse:,.2f}', f'{r2:.4f}']
})
print('=== Model Evaluation Metrics ===')
print(metrics.to_string(index=False))

## Step 5: Actual vs Predicted Chart

In [ ]:
train_pred = model.predict(X_train)
test_pred  = model.predict(X_test)
months     = monthly['YearMonth'].values
actuals    = monthly['MonthlyRevenue'].values

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')

ax.plot(months, actuals, color='#6C63FF', linewidth=2.2, label='Actual Revenue', zorder=3)
ax.fill_between(months, actuals, alpha=0.12, color='#6C63FF')
ax.plot(months[:split_idx], train_pred, '--', color='#43B6C8',
        linewidth=1.8, label='Train Fit')
ax.plot(months[split_idx:], test_pred, '--', color='#FF6B6B',
        linewidth=2.0, label='Test Prediction')
ax.axvline(x=months[split_idx], color='#FFB347', linewidth=1.5,
           linestyle=':', alpha=0.85, label='Train/Test Split')

ax.set_title('Actual vs Predicted Monthly Revenue', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(fontsize=10)

# Annotate R² on chart
ax.text(0.02, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes,
        fontsize=11, color='#333', verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', 'actual_vs_predicted.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step 6: 6-Month Future Forecast

In [ ]:
last_idx   = monthly['TimeIndex'].max()
last_month = pd.to_datetime(monthly['YearMonth'].iloc[-1])

future_rows = []
for i in range(1, 7):
    fd = last_month + pd.DateOffset(months=i)
    m  = fd.month
    future_rows.append({
        'YearMonth': fd.strftime('%Y-%m'),
        'TimeIndex': last_idx + i,
        'Month': m,
        'Quarter': (m - 1) // 3 + 1,
        'MonthSin': np.sin(2 * np.pi * m / 12),
        'MonthCos': np.cos(2 * np.pi * m / 12),
    })

future_df = pd.DataFrame(future_rows)
future_df['ForecastRevenue'] = model.predict(future_df[feature_cols].values).clip(min=0)
print('6-Month Forecast:')
print(future_df[['YearMonth', 'ForecastRevenue']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('#f8f9fa')

ax.plot(months, actuals, color='#6C63FF', linewidth=2.2, label='Historical Revenue')
ax.fill_between(months, actuals, alpha=0.12, color='#6C63FF')
ax.plot(future_df['YearMonth'], future_df['ForecastRevenue'],
        'o--', color='#56D79E', linewidth=2.2, markersize=7, label='6-Month Forecast')
ax.fill_between(future_df['YearMonth'], future_df['ForecastRevenue'],
                alpha=0.18, color='#56D79E')
ax.plot([months[-1], future_df['YearMonth'].iloc[0]],
        [actuals[-1], future_df['ForecastRevenue'].iloc[0]],
        '--', color='#aaa', linewidth=1.4)
ax.axvline(x=months[-1], color='#FFB347', linewidth=1.5,
           linestyle=':', alpha=0.9, label='Forecast Start')

ax.set_title('Revenue Forecast — Next 6 Months', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.yaxis.grid(True, color='#ddd', linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join('..', 'visualizations', 'revenue_forecast.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Total 6-Month Forecast Revenue: £{future_df["ForecastRevenue"].sum():,.2f}')

## ✅ Summary

The Linear Regression model with seasonal features provides a reliable revenue trend forecast.

**Next**: `05_business_insights.ipynb`